# Baum (tree-drawing) test — automatic feature extraction

This notebook runs `features.py` from the repository **baum-drawing-features** on your own scanned drawings, entirely inside Google Colab (nothing is installed on your computer).

**Steps**
1. Run the first cell to set up (about 1 minute).
2. Run the second cell and upload your scanned drawings (PNG/JPG/TIFF). You can also skip the upload and try the three synthetic examples.
3. Run the remaining cells to extract features, check the binarised images and download the results.

Scan so that the tree spans at least ~600 px on its long side; keep scanner settings constant within a study. See the README for details.

*Privacy note:* uploaded files are stored in a temporary Colab session that is deleted when the session ends. Do not upload identifiable material unless your data-handling rules allow it.

In [ ]:
#@title 1. Set up (clone the repository and install dependencies)
!git clone -q https://github.com/jmaruta/baum-drawing-features.git
%cd baum-drawing-features
!pip install -q -r requirements.txt
print('ready')

In [ ]:
#@title 2. Upload your drawings (or leave empty to use the synthetic examples)
import os, shutil
from google.colab import files
os.makedirs('my_scans', exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join('my_scans', name), 'wb') as f:
        f.write(data)
if not uploaded:
    for f in os.listdir('examples'):
        if f.lower().endswith('.png'):
            shutil.copy(os.path.join('examples', f), 'my_scans')
    print('No upload: using the synthetic examples.')
print(sorted(os.listdir('my_scans')))

In [ ]:
#@title 3. Extract features
!python features.py my_scans my_features.csv my_debug
import pandas as pd
df = pd.read_csv('my_features.csv')
cols = ['ID','res_long','ink_level','ink_dark','stroke_w_rel','jagged','ink_len','junctions_per_len','small_loops','crown_env_cov','trunk_crossings','ground_auto']
df[cols].round(3)

In [ ]:
#@title 4. Check the binarised drawings (always do this)
import matplotlib.pyplot as plt, cv2, os
names = sorted(os.listdir('my_debug'))
n = len(names); cols_ = min(4, n); rows_ = -(-n // cols_)
fig, axes = plt.subplots(rows_, cols_, figsize=(3.2*cols_, 3.6*rows_)); axes = [axes] if n == 1 else list(axes.flat)
for ax, nm in zip(axes, names):
    ax.imshow(cv2.imread(os.path.join('my_debug', nm), 0), cmap='gray'); ax.set_title(nm, fontsize=8); ax.axis('off')
for ax in axes[n:]: ax.axis('off')
plt.tight_layout(); plt.show()
print('If lines are missing, the scan is too faint or low-resolution; if printed text remains, crop the image closer to the tree.')

In [ ]:
#@title 5. Download the results
import shutil
from google.colab import files
shutil.make_archive('my_debug', 'zip', 'my_debug')
files.download('my_features.csv')
files.download('my_debug.zip')

## Column guide (short)

| column | meaning |
|---|---|
| `res_long` | long side of the input image in pixels (quality check; aim for ≥ 600) |
| `ink_level` | darkness of the darkest lines (quality check; very low values = faint scan) |
| `ink_dark`, `stroke_w_rel`, `stroke_w_cv`, `fill_ratio` | line darkness, relative stroke width and its variability, shading |
| `jagged`, `turn_hf`, `gaps_per_len`, `short_seg_frac`, `frag_small`, `n_strokes` | line irregularity and fragmentation |
| `ink_len`, `ink_cover`, `endpoints_per_len`, `junctions_per_len`, `small_loops` | amount of drawing, branching, small enclosed loops |
| `crown_env_cov`, `crown_env_md`, `crown_env_frac`, `crown_hole_seal` | crown closure |
| `trunk_crossings`, `crown_trunk_ratio`, `ground_auto`, `lr_asym`, `ink_top_half`, `centroid_y`, `aspect_hw` | trunk lines, proportions, ground line, layout |

Full definitions: Table 2 of the paper and the docstrings in `features.py`.